In [ ]:
from pathlib import Path
import pandas as pd

now_stats_folder = Path("provided/now_stats")
usage_stats_file = Path("provided/usage_stats.tsv")
precision_sample_file = Path("provided/precision_sample.tsv")

# Recall

In [ ]:
files = ["lemur_300_single.txt", "lemur_300_multi.txt", "lemur_1000_single.txt", "lemur_1000_multi.txt"]
# 10000 was used for the thesis as retrieval was capped at the same limit
limit = 10000

df_list = []
replace_lookup = {
    "to come back to haunt #": "to come back to haunt someone",
    "feel #'s pain": "feel someone's pain",
    "feel # pain": "feel someone's pain",
    "-ista": "ista",
}

# Read NOW webinterface stats
for file in files:
    key = file.replace("/", " ").replace("_", " ").replace(".txt", "").lower()
    path = now_stats_folder.joinpath(file)
    df_web = pd.read_csv(path, delimiter="\t", names=["headword", "count_web_exact", "count_web"])

    df_web["count_web"] = df_web["count_web count_web_exact".split()].max(axis=1)
    if limit > -1:
        df_web["count_web"] = df_web["count_web"].apply(lambda x: min(x, limit))
    df_web = df_web["headword count_web".split()]
    df_web["source"] = key.title()

    df_list.append(df_web)

df = pd.concat(df_list)

# Read latest retrieval stats
df_latest = pd.read_csv(usage_stats_file, delimiter="\t")
df_latest = df_latest["lemma found".split()]
df_latest = df_latest.rename(columns={"lemma": "headword", "found": "count_retrieval"})

# Map back to original headwords for placeholder/suffix lemmas
df_latest["headword"] = df_latest["headword"].replace(replace_lookup)
df_latest = df_latest.groupby("headword", as_index=False).sum()

# Merge stats on headword
df_merge = pd.merge(df, df_latest, how="left", on="headword")
df_merge = df_merge.fillna(0)
if limit > -1:
    df_merge["count_retrieval"] = df_merge["count_retrieval"].apply(lambda x: min(x, limit))
df_merge.set_index("headword", inplace=True)
df_merge["recall_raw"] = df_merge["count_retrieval"] / df_merge["count_web"] * 100

# Recall limited at 100, if more was found by our retrieval than web interface
df_merge["recall"] = df_merge["recall_raw"].apply(lambda x: min(x, 100))

# Print result
print(df_merge)
print()

In [ ]:
# Print each table from thesis
for source in df_merge["source"].unique():
    table = df_merge[df_merge["source"] == source]
    table = table.reset_index(inplace=False)
    table = table["headword count_web count_retrieval recall".split()]
    print(source)
    print(table)
    print()

In [ ]:
# Print overall stats
print("Overall Recall")
# print(f"Mean: {df_merge['recall'].mean():.1f}")
print(f"Median: {df_merge['recall'].median():.1f}")

In [ ]:
# Print stats for each subcategory
for criterion in ["300", "1000", "single", "multi", "300 single", "300 multi", "1000 single", "1000 multi"]:
    df = df_merge[df_merge["source"].str.contains(criterion, case=False)]
    print(criterion.title())
    # print(f"Mean Recall: {df['recall'].mean():.1f}")
    print(f"Median Recall: {df['recall'].median():.1f}")
    print()

# Precision

In [ ]:
df = pd.read_csv(precision_sample_file, delimiter="\t")
valid = len(df[df["valid"] == 1])
total = len(df)

precision = valid / total * 100
print("Precision")
print(f"{precision:.1f}")